# Voice Screening Agent - Colab runner

A voice agent that runs a short senior .NET technical screening end to end:
speak an answer, it transcribes (Egyptian Arabic + English code-mixing handled),
retrieves the relevant rubric criteria, **decides whether to ask one clarifying
follow-up**, then scores 1-5 and speaks the result back in your language.

Everything is open-source and self-hosted. No API keys.

**Before you start:** set the runtime to a GPU.
`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`

Then `Runtime -> Run all`. First run takes ~10-12 minutes, almost all of it
downloading ~10 GB of model weights.

The **last** cell launches the UI and prints a public `https://....gradio.live`
link. It is last on purpose: launching the UI blocks forever, so anything below
it would never run during `Run all`. The tests and the quality gate come first,
which also means that by the time you have a link, you already know the pipeline
works.


## 1. Confirm the GPU

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU"
)
print(f"\n{torch.cuda.get_device_name(0)} | "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Clone the repository

In [ ]:
import os, pathlib

REPO_URL = "https://github.com/karimgamalmahmoud/Voice-Agentic-Systems.git"
REPO_DIR = pathlib.Path("/content/Voice-Agentic-Systems")

if REPO_DIR.exists():
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("\nWorking directory:", os.getcwd())

## 3. Install Python dependencies

Torch is deliberately left alone - Colab's build is already CUDA-matched, and
reinstalling it is the fastest way to break the runtime.

If pip prints a "You must restart the runtime" banner, do it, then
`Runtime -> Run all` again. Cell 2 is a no-op on the second pass.

In [ ]:
!pip install -q -r requirements.txt

# Colab preinstalls versions that can shadow the ones we need; confirm the
# important three actually imported at the versions we expect.
import transformers, gradio, sentence_transformers
print("transformers", transformers.__version__)
print("gradio", gradio.__version__)
print("sentence-transformers", sentence_transformers.__version__)

## 4. Install and start Ollama

Ollama serves the LLM behind an OpenAI-compatible API. It runs as a separate
process, which keeps the LLM's dependencies completely isolated from the
torch / Whisper / TTS stack above - the single biggest cause of Colab installs
falling over.

In [ ]:
# Ollama's installer unpacks a zstd-compressed archive and the Colab image does
# not ship zstd. Without this the install dies with
# "ERROR: This version requires zstd for extraction".
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)

!curl -fsSL https://ollama.com/install.sh | sh

# Fail here rather than three cells later with a confusing connection error
# against 127.0.0.1:11434, which points at entirely the wrong problem.
import shutil
assert shutil.which("ollama"), "Ollama did not install - check the output above"
!ollama --version

In [ ]:
import subprocess, time, os, requests

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

# Start the server detached; Colab reaps foreground background jobs between cells.
log = open("/content/ollama.log", "w")
server = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)

for attempt in range(60):
    try:
        if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok:
            print(f"Ollama up after {attempt + 1}s")
            break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/ollama.log").read()[-2000:])
    raise RuntimeError("Ollama did not start - log above")

### Pull the model

`qwen2.5:7b-instruct` (~4.7 GB). Chosen for two reasons: it is the strongest
Arabic model that fits a free T4 alongside Whisper and BGE-M3, and it is
reliable at emitting the structured JSON the coverage and scoring stages
depend on.

In [ ]:
!ollama pull qwen2.5:7b-instruct
!ollama list

## 5. Unit tests

CPU-only, a few seconds. Covers the follow-up decision policy, corpus chunking
and code-mixed language detection. If these fail, stop - something is wrong with
the checkout, not with your GPU.

In [ ]:
!python -m pytest tests/ -q

## 6. Preload the speech and embedding models

Whisper large-v3 (~3 GB) and BGE-M3 (~2.2 GB). Doing it now rather than on the
first click means the live demo is not sitting through a download. Expect
~4-6 minutes.

In [ ]:
import sys
sys.path.insert(0, "/content/Voice-Agentic-Systems/src")

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from voice_agent.agent import ScreeningAgent

agent = ScreeningAgent()
agent.warm_up()          # Whisper large-v3 + BGE-M3
ok, msg = agent.llm.health()
print("\nLLM:", msg)
assert ok, msg

## 7. Transcription check

The riskiest part of this stack is Egyptian Arabic mixed with English technical
terms, so it is worth eyeballing the transcripts before anything else. Two of
the three provided samples are code-mixed.

Look for: Arabic script for the Arabic, and English technical terms
("async", "EF Core", "thread pool") surviving in Latin script rather than being
transliterated into Arabic.

In [ ]:
!python scripts/transcribe_samples.py

## 8. Quality gate - the full pipeline over all three samples

This is the end-to-end proof, and a required deliverable. Each provided sample
runs transcribe -> retrieve -> assess coverage -> branch -> score, and the
invariants are checked: the two irrelevant reference notes stay out of
retrieval, Arabic in produces Arabic out, and scores have not drifted from the
stored baseline.

Writes `docs/QUALITY_GATE_RESULTS.md`. First run creates the baseline, so it
cannot fail on drift; later runs compare against it.

`--no-speak` skips TTS here to keep it quick - the speak-back is exercised in
the UI below.

In [ ]:
!python scripts/run_quality_gate.py --no-speak

In [ ]:
# Read the generated report inline.
from IPython.display import Markdown, display
display(Markdown(open("docs/QUALITY_GATE_RESULTS.md", encoding="utf-8").read()))

### Commit the results back (optional)

The quality-gate report is a required submission artifact. Easiest path is to
download it and commit from your machine.

In [ ]:
from google.colab import files
files.download("docs/QUALITY_GATE_RESULTS.md")

## 9. Check the Arabic speak-back

TTS is the weakest component in an all-open-source stack, so confirm it makes
sound before relying on it in a live demo. If this cell produces silence, the
loop still works - the UI falls back to text - but you will want to know now.

In [ ]:
from IPython.display import Audio, display

sample = agent.tts.synthesize("تقييمك 4 من 5. إجابة كويسة بس ناقصها تفاصيل.", "ar")
if sample is None:
    print("Arabic TTS unavailable - the UI will show text only.")
else:
    rate, wav = sample
    print(f"Arabic OK: {len(wav) / rate:.1f}s")
    display(Audio(wav, rate=rate))

## 10. Launch the app  ← last cell, blocks while running

Prints a public `gradio.live` URL. Open it in a **new tab** - the microphone
works through that tunnel, which is what makes this runnable with no local
install. Grant mic permission on the `gradio.live` origin, not on the Colab tab.

The link stays alive while this cell runs. Stop the cell to shut it down.

In [ ]:
from voice_agent.app import build_ui
import voice_agent.app as app_module

app_module.AGENT = agent      # reuse the models already loaded in cell 6
build_ui().launch(share=True)

---

### Troubleshooting

**`ERROR: This version requires zstd for extraction`** - the Ollama installer
unpacks a zstd archive and the Colab image does not ship zstd. Cell 4 installs
it first; if you are installing by hand, run `!apt-get -qq install -y zstd`
before the install script.

**Gradio link not appearing** - the last cell must stay running. If it errored,
rerun the Ollama start cell; Colab sometimes reaps the process.

**`model not found`** - rerun the `ollama pull` cell. The pull silently no-ops
if the server was not up yet.

**CUDA out of memory** - `Runtime -> Restart session`, then run all again.
Whisper large-v3, BGE-M3 and Qwen-7B together sit around 11 GB of the T4's
15 GB, so there is headroom, but a stale session can hold weights. Ollama
unloads the LLM after five idle minutes and reloads on demand, which is why the
first request after a pause is slow.

**Arabic speak-back is silent** - MMS-TTS needs romanized input via the `uroman`
package. If it failed to install, TTS degrades to text-only by design and the
rest of the loop is unaffected.

**Microphone blocked** - the browser needs permission on the `gradio.live`
origin, not on the Colab tab. Look for the mic icon in the address bar.
